# Risk assessment for relative drought — Kazakhstan

- Adapted from the CLIMAAX [Handbook](https://handbook.climaax.eu/) and [DROUGHTS](https://github.com/CLIMAAX/DROUGHTS) GitHub repository.
- Methodology: Carrão et al. (2016) [doi:10.1016/j.gloenvcha.2016.04.012](https://doi.org/10.1016/j.gloenvcha.2016.04.012)
- Composite index weighting: Cherchye et al. (2007) Benefit-of-the-Doubt DEA approach (replaces the standard input-oriented DEA used in the original CLIMAAX workflow).

**Key differences from the original CLIMAAX workflow:**
- Spatial units are Kazakhstan **LVL3 districts** (227 units) rather than EU NUTS3 regions.
- District-level granularity ensures sufficient sample size for statistically stable BoD-DEA optimisation.
- Composite keys (`district_name | oblast_name`) are used throughout to handle non-unique district names across oblasts.
- Hazard scores are loaded from the companion hazard notebook output; values are in [0, 1] and used directly without additional normalisation.
- Risk classification uses Fisher-Jenks natural breaks, consistent with the hazard notebook.
- SSP5-8.5 is not included; SSP1-2.6 and SSP3-7.0 are processed (near-future 2031–2060 and far-future 2071–2100).

## Cell 0 — Imports & configuration

All paths, scenario labels, and colour schemes are defined here.  
Edit `SHAPEFILE_LVL3` and the data-root paths to match your local environment.

In [5]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import rasterstats
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib.colors import BoundaryNorm
from pathlib import Path
from scipy.optimize import linprog

try:
    from jenkspy import jenks_breaks
    HAS_JENKSPY = True
except ImportError:
    HAS_JENKSPY = False
    print('WARNING: jenkspy not installed — falling back to quantile breaks.'
          '  Install with: pip install jenkspy')

# ── Shapefile ─────────────────────────────────────────────────────────────
SHAPEFILE_LVL3 = Path(
    r'C:\Users\dauzo\Models\crabook-kazakhstan\crabook'
    r'\KAZ_BORDER_VERSION_1\LVL3\SHP_LVL3\KAZ_OSM_BORDER_LVL3.shp'
)  # <-- update to your local shapefile path
REGION_NAME_FIELD = 'name_en'
OBLAST_NAME_FIELD = 'oblast_en'

# ── Directory structure ───────────────────────────────────────────────────
BASE_DIR      = Path('./data/isimip3b')
PROCESSED_DIR = BASE_DIR / 'processed'
OUTPUT_DIR    = PROCESSED_DIR / 'outputs_hazards'
RISK_DIR      = PROCESSED_DIR / 'outputs_risk_districts'
RISK_DIR.mkdir(parents=True, exist_ok=True)

# ── Raw data paths ────────────────────────────────────────────────────
# Source: WorldPop (https://www.worldpop.org/) — download KAZ 100m population raster
WORLDPOP_FILE = BASE_DIR / 'worldpop' / 'kaz_pop_2025_100m.tif'  # <-- update to your downloaded file path

DATA_ROOT = Path(
    r'C:\Users\dauzo\Models\crabook-kazakhstan\crabook'
    r'\notebooks\workflows\DROUGHTS\01_relative_drought\data'
)  # <-- update to your local data root path

# Source: SPAM 2020 (https://www.mapspam.info/) — download harvested-area TIFs
SPAM_DIR = DATA_ROOT / 'Crop Production Statistics Data'

# Source: GLW4 2020 (https://dataverse.harvard.edu/dataverse/glw) — download GLEAM3 ALL-LU raster
GLW_FILE = DATA_ROOT / 'GLW4-2020.D-DA.GLEAM3-ALL-LU.tif'  # <-- update to your downloaded file path

# Source: GRIP4 roads (https://www.globio.info/download-grip-dataset) — download Region 5 shapefile
GRIP_FILE = DATA_ROOT / 'Roads' / 'Roads_KZ_main' / 'Roads_KZ_main.shp'  # <-- update to your downloaded file path

# Source: Aqueduct 4.0 (https://www.wri.org/aqueduct) — download baseline annual CSV
AQUEDUCT_CSV = (
    DATA_ROOT
    / r'Aqueduct40_waterrisk_download_Y2023M07D05\CVS'
    / 'Aqueduct40_baseline_annual_y2023m07d05.csv'
)  # <-- update to your downloaded file path

# Source: GDL subnational HDI (https://globaldatalab.org/shdi/) — download CSV
GDL_FILE = DATA_ROOT / 'GDL-Subnational-HDI-data.csv'  # <-- update to your downloaded file path

# Source: GDL subnational GDP (https://globaldatalab.org/areadata/) — download GDI CSV
GDL_GDP_FILE = DATA_ROOT / 'GDL-Subnational-GDI-data.csv'  # <-- update to your downloaded file path

# ── Scenarios ─────────────────────────────────────────────────────────────
SCENARIOS = ['historic', 'ssp126_nf', 'ssp126_ff', 'ssp370_nf', 'ssp370_ff']
SCENARIO_TITLES = {
    'historic':  'Historical\n(1981\u20132014)',
    'ssp126_nf': 'SSP1-2.6\nNear-future (2031\u20132060)',
    'ssp126_ff': 'SSP1-2.6\nFar-future (2071\u20132100)',
    'ssp370_nf': 'SSP3-7.0\nNear-future (2031\u20132060)',
    'ssp370_ff': 'SSP3-7.0\nFar-future (2071\u20132100)',
}

# ── Visualisation palette (CLIMAAX standard) ────────────────────────────────────
RISK_COLORS = ['#2b83ba', '#abdda4', '#ffffbf', '#fdae61', '#d7191c']
RISK_LABELS = ['Very Low', 'Low', 'Medium', 'High', 'Very High']

# ── DEA diagnostic flag ────────────────────────────────────────────────────────────
evaluateDEA = False  # set True to print scatter plots for DEA quality evaluation

print('Configuration ready.')

Configuration ready.


## Cell 1 — Load district shapefile

The LVL3 shapefile provides 227 districts (rayon level).  
A composite key `district_name | oblast_name` is created to uniquely identify districts whose names are repeated across different oblasts.  
All subsequent indicator series use this `_key` as their index.

In [2]:
districts = gpd.read_file(SHAPEFILE_LVL3).to_crs(epsg=4326)
districts = districts.rename(columns={
    REGION_NAME_FIELD: 'district_name',
    OBLAST_NAME_FIELD: 'oblast_name',
})
districts['district_name'] = districts['district_name'].str.strip()
districts['oblast_name']   = districts['oblast_name'].str.strip()

# Composite key — unique even when two districts share the same English name
districts['_key'] = districts['district_name'] + ' | ' + districts['oblast_name']

# District area in km² (equal-area CRS)
districts_ea = districts.to_crs(epsg=8857)
districts['area_km2'] = districts_ea.geometry.area / 1e6

# Unique oblast list — required by the Aqueduct BWS processing
oblasts = (
    districts[['oblast_name']]
    .drop_duplicates()
    .reset_index(drop=True)
)

# districts_unique: one row per composite _key, used for indicator extraction
# (may differ from len(districts) if the shapefile has multi-part geometries)
districts_unique = districts.drop_duplicates(subset=['_key'], keep='first').copy()

print(f'Districts (LVL3) : {len(districts)}')
print(f'Unique _keys     : {len(districts_unique)}')
print(f'Oblasts          : {len(oblasts)}')
print(f'\nSample _keys:')
print(districts[['_key', 'area_km2']].head(8).to_string(index=False))

Districts (LVL3) : 227
Unique _keys     : 227
Oblasts          : 20

Sample _keys:
                                      _key     area_km2
Zhanybek District | West Kazakhstan Region  8204.158281
       Kurmangazy district | Atyrau Region 21844.277726
            Ïnder district | Atyrau Region 10970.139750
           Isatay District | Atyrau Region 14910.226731
        Kyzylkoga District | Atyrau Region 24922.318537
            Makat District | Atyrau Region  4847.827495
        Makhambet District | Atyrau Region  9763.898910
          Zhylyoi District | Atyrau Region 30114.181063


## Cell 2 — Load district-level hazard scores

Hazard scores are produced by the companion **Hazard assessment** notebook.  
The script tries to load a consolidated CSV first; if absent it merges per-scenario files.  
The hazard `district` column must match the `_key` composite key used for all other indicators.

In [3]:
hazard_path = OUTPUT_DIR / 'droughthazard_KZ_all_scenarios.csv'

if hazard_path.exists():
    hazard_all = pd.read_csv(hazard_path)
    print(f'Loaded consolidated hazard file: {hazard_path.name}')
else:
    print('Consolidated file not found — merging per-scenario CSVs...')
    hazard_file_map = {
        'historic':  'droughthazard_KZ_historic.csv',
        'ssp126_nf': 'droughthazard_KZ_ssp126_nf.csv',
        'ssp126_ff': 'droughthazard_KZ_ssp126_ff.csv',
        'ssp370_nf': 'droughthazard_KZ_ssp370_nf.csv',
        'ssp370_ff': 'droughthazard_KZ_ssp370_ff.csv',
    }
    hazard_all = None
    for scen, fname in hazard_file_map.items():
        fpath = OUTPUT_DIR / fname
        if not fpath.exists():
            print(f'  SKIP {fname} — not found')
            continue
        df = pd.read_csv(fpath)
        score_col = 'hazard_raw' if 'hazard_raw' in df.columns else scen
        df = df.rename(columns={score_col: scen})
        if hazard_all is None:
            hazard_all = df[['district', scen]].copy()
        else:
            hazard_all = hazard_all.merge(df[['district', scen]], on='district', how='outer')
        print(f'  Loaded {fname}')

    if hazard_all is None:
        raise FileNotFoundError(f'No hazard CSV files found in {OUTPUT_DIR}')

# Verify key alignment
hazard_keys    = set(hazard_all['district'])
shapefile_keys = set(districts['_key'])
n_missing = len(shapefile_keys - hazard_keys)
if n_missing:
    print(f'WARNING: {n_missing} shapefile districts not found in hazard data')
else:
    print('All district keys matched successfully.')

avail_scens = [s for s in SCENARIOS if s in hazard_all.columns]
print(f'\nAvailable scenarios : {avail_scens}')
print(f'Districts in hazard : {len(hazard_all)}')
print(hazard_all[['district'] + avail_scens].head(5).to_string(index=False))
print('>>>>> Hazard data loaded.')

Loaded consolidated hazard file: droughthazard_KZ_all_scenarios.csv

Available scenarios : ['historic', 'ssp126_nf', 'ssp126_ff', 'ssp370_nf', 'ssp370_ff']
Districts in hazard : 186
                                  district  historic  ssp126_nf  ssp126_ff  ssp370_nf  ssp370_ff
               Abay District | Abay Region     0.785      0.589      0.701      0.655      0.685
          Abay District | Karaganda Region     0.874      0.695      0.721      0.724      0.750
Aiyrtau District | North Kazakhstan Region     0.653      0.697      0.744      0.663      0.810
            Akkol District | Akmola Region     0.826      0.736      0.732      0.721      0.706
                    Aksu | Pavlodar Region     0.793      0.699      0.633      0.735      0.700
>>>>> Hazard data loaded.


## Cell 3 — Exposure indicators

Four exposure indicators following Carrão et al. (2016):

| ID | Indicator | Source | Unit |
|---|---|---|---|
| E1 | Population density | WorldPop 2025 | persons/km² |
| E2 | Cropland density | SPAM 2020 | ha/km² |
| E3 | Livestock density | GLW4-2020 | head/km² |
| E4 | Baseline water stress | Aqueduct 4.0 | score 0–5 |

All series are indexed by the composite `_key` from the start to avoid ambiguity.

In [6]:
# ── E1: WorldPop 2025 — population density ────────────────────────────────────────
print('[E1] WorldPop — population density ...')
with rasterio.open(WORLDPOP_FILE) as src:
    wp_nodata = src.nodata

pop_stats = rasterstats.zonal_stats(
    districts, str(WORLDPOP_FILE),
    stats=['sum'], nodata=wp_nodata, geojson_out=False
)
pop_series = pd.Series(
    [s['sum'] if s['sum'] is not None else 0.0 for s in pop_stats],
    index=districts['_key'],
    name='population'
)
pop_density = (
    pop_series / districts.set_index('_key')['area_km2']
).rename('pop_density')
print(f'  Total KZ population : {pop_series.sum()/1e6:.2f} M')
print(f'  Max density         : {pop_density.max():.1f} p/km\u00b2  ({pop_density.idxmax()})')


# ── E2: SPAM 2020 — cropland density ─────────────────────────────────────────────
spam_cache = PROCESSED_DIR / 'spam_cropland_kaz.csv'

if spam_cache.exists():
    spam_df     = pd.read_csv(spam_cache, index_col=0)
    spam_series = spam_df.iloc[:, 0].rename('cropland_ha')
    if '|' not in str(spam_series.index[0]):
        name_to_key = districts_unique.set_index('district_name')['_key']
        spam_series.index = spam_series.index.map(name_to_key)
        spam_series = spam_series.dropna()
    print(f'[E2] SPAM loaded from cache: {spam_cache.name}')
else:
    print('[E2] SPAM — summing harvested area across all crops ...')
    crop_tifs = sorted([
        f for f in SPAM_DIR.rglob('*.tif')
        if '_H_' in f.name and f.name.endswith('_A.tif')
    ])
    print(f'  Found {len(crop_tifs)} crop TIFs')
    spam_vals = np.zeros(len(districts))
    for tif in crop_tifs:
        with rasterio.open(tif) as src:
            nd = src.nodata if src.nodata is not None else -9999.0
        st = rasterstats.zonal_stats(
            districts, str(tif), stats=['sum'], nodata=nd, geojson_out=False
        )
        spam_vals += np.array([
            s['sum'] if (s['sum'] is not None and s['sum'] > 0) else 0.0
            for s in st
        ])
    spam_series = pd.Series(spam_vals, index=districts['_key'], name='cropland_ha')
    spam_series.to_csv(spam_cache)
    print(f'  Saved cache: {spam_cache.name}')

cropland_density = (
    spam_series / districts.set_index('_key')['area_km2']
).rename('cropland_ha_per_km2')
print(f'  KZ total cropland : {spam_series.sum()/1e6:.3f} M ha')


# ── E3: GLW4-2020 — livestock density ────────────────────────────────────────────
glw_cache = PROCESSED_DIR / 'glw_livestock_kaz.csv'

if glw_cache.exists():
    glw_df           = pd.read_csv(glw_cache, index_col=0)
    livestock_series = glw_df.iloc[:, 0].rename('livestock_density')
    if '|' not in str(livestock_series.index[0]):
        name_to_key = districts_unique.set_index('district_name')['_key']
        livestock_series.index = livestock_series.index.map(name_to_key)
        livestock_series = livestock_series.dropna()
    print(f'[E3] GLW loaded from cache: {glw_cache.name}')
else:
    print('[E3] GLW — aggregating livestock density ...')
    with rasterio.open(GLW_FILE) as src:
        glw_nodata = src.nodata
    glw_stats = rasterstats.zonal_stats(
        districts, str(GLW_FILE),
        stats=['mean'], nodata=glw_nodata, geojson_out=False
    )
    livestock_series = pd.Series(
        [s['mean'] if s['mean'] is not None else 0.0 for s in glw_stats],
        index=districts['_key'],
        name='livestock_density'
    )
    livestock_series.to_csv(glw_cache)
    print(f'  Saved cache: {glw_cache.name}')

print(f'  Max livestock density : {livestock_series.max():.2f}  ({livestock_series.idxmax()})')


# ── E4: Aqueduct 4.0 BWS — oblast → district inheritance ───────────────────────
print('[E4] Aqueduct BWS — building district-level scores ...')

aq_cols = ['name_0', 'name_1', 'area_km2', 'bws_score']
aq_csv  = pd.read_csv(AQUEDUCT_CSV, usecols=aq_cols, low_memory=False)
kaz_aq  = aq_csv[aq_csv['name_0'] == 'Kazakhstan'].copy()
kaz_aq['bws_score'] = pd.to_numeric(kaz_aq['bws_score'], errors='coerce')
kaz_aq.loc[kaz_aq['bws_score'] == -1, 'bws_score'] = 0.0
kaz_aq  = kaz_aq[kaz_aq['bws_score'] >= 0].dropna(subset=['bws_score'])
kaz_aq['area_km2'] = pd.to_numeric(kaz_aq['area_km2'], errors='coerce').fillna(0)

aq_oblast = (
    kaz_aq.groupby('name_1')
    .apply(lambda g: np.average(g['bws_score'], weights=g['area_km2'].clip(lower=0.01)))
    .rename('bws_score')
)

AQ_TO_OBLAST = {
    'Aqmola':          'Akmola Region',
    'Aqt\u00f6be':         'Aqt\u00f6be region',
    'Almaty':          'Almaty Region',
    'Atyrau':          'Atyrau Region',
    'East Kazakhstan': 'East Kazakhstan Region',
    'Mangghystau':     'Mangystau Region',
    'North Kazakhstan': 'North Kazakhstan Region',
    'Pavlodar':        'Pavlodar Region',
    'Qaraghandy':      'Karaganda Region',
    'Qostanay':        'Kostanay Region',
    'Qyzylorda':       'Kyzylorda Region',
    'South Kazakhstan': 'Turkistan Region',
    'West Kazakhstan': 'West Kazakhstan Region',
    'Zhambyl':         'Jambyl Region',
}
bws_oblast = (
    aq_oblast.rename(index=AQ_TO_OBLAST)
    .reindex(oblasts['oblast_name'])
)

INHERIT = {
    'Abay Region':   'East Kazakhstan Region',
    'Jetisu Region': 'Almaty Region',
    'Ulytau Region': 'Karaganda Region',
    'Almaty':        'Almaty Region',
    'Astana':        'Akmola Region',
    'Shymkent':      'Turkistan Region',
}
for ob, parent in INHERIT.items():
    if ob in bws_oblast.index and pd.isna(bws_oblast.get(ob)):
        if parent in bws_oblast.index and not pd.isna(bws_oblast.get(parent)):
            bws_oblast[ob] = bws_oblast[parent]

n_miss = bws_oblast.isna().sum()
if n_miss:
    bws_oblast = bws_oblast.fillna(bws_oblast.median())
    print(f'  {n_miss} oblasts filled with median BWS')

bws_series = (
    districts_unique.set_index('_key')['oblast_name']
    .map(bws_oblast)
    .rename('bws_score')
)
n_miss2 = bws_series.isna().sum()
if n_miss2:
    print(f'  WARNING: {n_miss2} districts missing BWS — filling median')
    bws_series = bws_series.fillna(bws_series.median())

print(f'  BWS range: {bws_series.min():.3f} \u2013 {bws_series.max():.3f}')


# ── Save exposure cache ───────────────────────────────────────────────────────────────
exposure_df = pd.DataFrame({
    'pop_density':         pop_density,
    'cropland_ha_per_km2': cropland_density,
    'livestock_density':   livestock_series,
    'bws_score':           bws_series,
})
exposure_df.index.name = 'district'
exposure_df.to_csv(RISK_DIR / 'exposure_indicators_districts.csv')
print(f'\nExposure saved \u2192 exposure_indicators_districts.csv')
print(exposure_df.describe().round(3))
print('>>>>> Exposure indicators assembled.')

[E1] WorldPop — population density ...
  Total KZ population : 20.94 M
  Max density         : 6957.0 p/km²  (Auezov District | Almaty)
[E2] SPAM loaded from cache: spam_cropland_kaz.csv
  KZ total cropland : 19.460 M ha
[E3] GLW loaded from cache: glw_livestock_kaz.csv
  Max livestock density : 48.58  (Qarataw district | Shymkent)
[E4] Aqueduct BWS — building district-level scores ...
  BWS range: 0.620 – 4.929

Exposure saved → exposure_indicators_districts.csv
       pop_density  cropland_ha_per_km2  livestock_density  bws_score
count      227.000              227.000            227.000    227.000
mean       306.904               13.055              7.306      2.129
std       1002.628               16.158              9.303      1.226
min          0.102                0.000              0.000      0.620
25%          1.735                0.766              2.717      1.223
50%          4.267                4.675              3.719      1.604
75%         38.014               21.151   

## Cell 4 — Vulnerability indicators

Three vulnerability indicators following Carrão et al. (2016):

| ID | Indicator | Source | Direction |
|---|---|---|---|
| V1 | Human Development Index (HDI) | GDL subnational 2022 | Inverted (high HDI → low vulnerability) |
| V2 | GDP per capita | GDL subnational (latest year) | Inverted |
| V3 | Road density | GRIP4 Region 5 | Inverted |

GDL provides Kazakhstan data at macro-region level (6 regions); these are mapped to oblasts and inherited by districts.  
All series are indexed by `_key`.

In [7]:
# ── V1: HDI ──────────────────────────────────────────────────────────────────────
print('[V1] HDI — loading from GDL ...')
gdl_raw = pd.read_csv(GDL_FILE, low_memory=False)
kaz_gdl = gdl_raw[
    (gdl_raw['ISO_Code'] == 'KAZ') & (gdl_raw['Level'] == 'Subnat')
].copy()
HDI_YEAR = '2022'
kaz_gdl  = kaz_gdl[['GDLCODE', 'Region', HDI_YEAR]].rename(columns={HDI_YEAR: 'hdi'})
gdl_hdi  = kaz_gdl.set_index('GDLCODE')['hdi']

OBLAST_TO_GDL = {
    'Almaty':                  'KAZr101',
    'Almaty Region':           'KAZr102',
    'Jambyl Region':           'KAZr102',
    'Kyzylorda Region':        'KAZr102',
    'Turkistan Region':        'KAZr102',
    'Jetisu Region':           'KAZr102',
    'Shymkent':                'KAZr102',
    'Aqt\u00f6be region':          'KAZr103',
    'Atyrau Region':           'KAZr103',
    'Mangystau Region':        'KAZr103',
    'West Kazakhstan Region':  'KAZr103',
    'Karaganda Region':        'KAZr104',
    'Ulytau Region':           'KAZr104',
    'Akmola Region':           'KAZr105',
    'Kostanay Region':         'KAZr105',
    'Pavlodar Region':         'KAZr105',
    'North Kazakhstan Region': 'KAZr105',
    'Astana':                  'KAZr105',
    'East Kazakhstan Region':  'KAZr106',
    'Abay Region':             'KAZr106',
}
hdi_oblast = pd.Series(
    {ob: gdl_hdi.get(gc, np.nan) for ob, gc in OBLAST_TO_GDL.items()},
    name='hdi'
)
hdi_series = (
    districts_unique.set_index('_key')['oblast_name']
    .map(hdi_oblast)
    .rename('hdi')
)
if hdi_series.isna().any():
    n = hdi_series.isna().sum()
    hdi_series = hdi_series.fillna(hdi_series.median())
    print(f'  {n} HDI values filled with median')
print(f'  HDI range: {hdi_series.min():.3f} \u2013 {hdi_series.max():.3f}')


# ── V2: GDP per capita ─────────────────────────────────────────────────────────────────
print('[V2] GDP per capita — loading from GDL ...')
gdl_gdp_raw = pd.read_csv(GDL_GDP_FILE, low_memory=False)
kaz_gdp     = gdl_gdp_raw[
    (gdl_gdp_raw['ISO_Code'] == 'KAZ') & (gdl_gdp_raw['Level'] == 'Subnat')
].copy()
year_cols = [c for c in kaz_gdp.columns if str(c).isdigit() and int(c) >= 2010]
GDP_YEAR  = str(max(int(y) for y in year_cols))
print(f'  Using GDP year: {GDP_YEAR}')
kaz_gdp   = kaz_gdp[['GDLCODE', 'Region', GDP_YEAR]].rename(columns={GDP_YEAR: 'gdp_pc'})
gdl_gdppc = kaz_gdp.set_index('GDLCODE')['gdp_pc']

gdp_oblast = pd.Series(
    {ob: gdl_gdppc.get(gc, np.nan) for ob, gc in OBLAST_TO_GDL.items()},
    name='gdp_pc'
)
gdp_series = (
    districts_unique.set_index('_key')['oblast_name']
    .map(gdp_oblast)
    .rename('gdp_pc')
)
if gdp_series.isna().any():
    n = gdp_series.isna().sum()
    gdp_series = gdp_series.fillna(gdp_series.median())
    print(f'  {n} GDP values filled with median')
print(f'  GDP/cap range: {gdp_series.min():.0f} \u2013 {gdp_series.max():.0f}')


# ── V3: GRIP4 Road density ─────────────────────────────────────────────────────
print('[V3] GRIP4 — computing road density per district ...')
grip_cache = RISK_DIR / 'grip4_road_density_districts.csv'

road_density = None
if grip_cache.exists():
    rd_df = pd.read_csv(grip_cache, index_col=0)
    if '|' in str(rd_df.index[0]):
        road_density = rd_df.iloc[:, 0].rename('road_density_km_per_km2')
        print(f'  Loaded from cache: {grip_cache.name}')
    else:
        print('  Cache uses old index format — recomputing ...')

if road_density is None:
    from shapely.geometry import box as sbox
    print('  Loading GRIP4 shapefile (may take ~1 min) ...')
    grip         = gpd.read_file(GRIP_FILE).to_crs(epsg=8857)
    districts_ea2 = districts.to_crs(epsg=8857)
    kaz_bbox     = districts_ea2.total_bounds
    grip_kz      = grip.cx[kaz_bbox[0]:kaz_bbox[2], kaz_bbox[1]:kaz_bbox[3]]
    grip_kz      = gpd.clip(grip_kz, districts_ea2.dissolve())
    print(f'  Road segments in KZ: {len(grip_kz):,}')
    grip_kz['length_km'] = grip_kz.geometry.length / 1000.0
    road_in_dist = gpd.sjoin(
        grip_kz[['length_km', 'geometry']],
        districts_ea2[['_key', 'geometry', 'area_km2']],
        how='left', predicate='intersects'
    )
    total_len    = road_in_dist.groupby('_key')['length_km'].sum()
    area         = districts_unique.set_index('_key')['area_km2']
    road_density = (total_len / area).reindex(districts_unique['_key']).fillna(0)
    road_density.name = 'road_density_km_per_km2'
    road_density.to_csv(grip_cache)
    print(f'  Saved: {grip_cache.name}')

print(f'  Road density range: {road_density.min():.4f} \u2013 {road_density.max():.4f} km/km\u00b2')


# ── Save vulnerability cache ─────────────────────────────────────────────────────────
vuln_df = pd.DataFrame({
    'hdi':                     hdi_series,
    'gdp_pc':                  gdp_series,
    'road_density_km_per_km2': road_density,
})
vuln_df.index.name = 'district'
vuln_df.to_csv(RISK_DIR / 'vulnerability_indicators_districts.csv')
print(f'\nVulnerability saved \u2192 vulnerability_indicators_districts.csv')
print(vuln_df.describe().round(3))
print('>>>>> Vulnerability indicators assembled.')

[V1] HDI — loading from GDL ...
  HDI range: 0.785 – 0.831
[V2] GDP per capita — loading from GDL ...
  Using GDP year: 2022
  GDP/cap range: 1 – 1
[V3] GRIP4 — computing road density per district ...
  Loaded from cache: grip4_road_density_districts.csv
  Road density range: 0.0242 – 6.5366 km/km²

Vulnerability saved → vulnerability_indicators_districts.csv
           hdi   gdp_pc  road_density_km_per_km2
count  227.000  227.000                  227.000
mean     0.801    0.998                    0.581
std      0.016    0.006                    1.015
min      0.785    0.992                    0.024
25%      0.785    0.992                    0.100
50%      0.794    0.999                    0.173
75%      0.816    0.999                    0.321
max      0.831    1.014                    6.537
>>>>> Vulnerability indicators assembled.


## Cell 5 — Benefit-of-the-Doubt DEA composite index

Implements the Cherchye et al. (2007) *Benefit of the Doubt* composite indicator as a replacement for the standard input-oriented DEA (`envelopmentpy`) used in the original CLIMAAX workflow (Carrão et al. 2016). The BoD formulation allows each district to receive the weighting vector most favourable to it:

$$CI_i = \max_{w \geq \varepsilon} \sum_j w_j x_{ij} \quad \text{s.t.} \quad \sum_j w_j x_{kj} \leq 1 \;\; \forall k$$

where $x_{ij}$ are min-max normalised indicator values floored at 0.01 following Carrão et al. (2016).

**Exposure** uses a single BoD-DEA over all four indicators.

**Vulnerability** follows the two-step structure of Carrão et al. (2016) Eq. 3: separate BoD-DEA runs per factor group (Social, Economic, Infrastructure), then arithmetic mean:

$$dv_i = \frac{Soc_i + Econ_i + Infr_i}{3}$$

Indicators signifying lower vulnerability with higher values (HDI, GDP/cap, road density) are **inverted** before DEA so that the composite always reads as *higher = worse*.

In [8]:
def minmax_norm(s: pd.Series) -> pd.Series:
    """Min-max normalise a Series to [0, 1]. Returns 0.0 if constant."""
    mn, mx = s.min(), s.max()
    if mx == mn:
        return pd.Series(0.0, index=s.index)
    return (s - mn) / (mx - mn)


def bod_dea(X: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    """
    Benefit-of-the-Doubt composite index for N regions x M indicators.
    Replaces the standard input-oriented DEA (envelopmentpy) used in the original
    CLIMAAX workflow; see Cherchye et al. (2007) for the BoD formulation.

    Parameters
    ----------
    X   : (N, M) array of normalised indicator values
    eps : lower bound on weights (prevents zero-weight degenerate solutions)

    Returns
    -------
    ci  : (N,) composite index
    """
    N, M = X.shape
    ci   = np.zeros(N)
    A_ub = X
    b_ub = np.ones(N)
    bounds = [(eps, 1.0)] * M

    for i in range(N):
        c   = -X[i, :]
        res = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method='highs')
        if res.success:
            ci[i] = -res.fun
        else:
            ci[i] = np.nan
    return ci


def compute_composite(series_dict: dict, invert: list = None) -> pd.Series:
    """
    Normalise indicators, optionally invert, apply 0.01 floor (Carrao et al. 2016),
    then compute BoD DEA.  Higher score = higher exposure/vulnerability.
    """
    if invert is None:
        invert = []
    names = list(series_dict.keys())
    idx   = list(series_dict.values())[0].index
    normed = []
    for nm in names:
        n = minmax_norm(series_dict[nm].reindex(idx))
        if nm in invert:
            n = 1.0 - n
        n = np.maximum(n, 0.01)  # floor applied after inversion, matching Carrao et al. (2016)
        normed.append(n.values)
    X  = np.column_stack(normed)
    ci = bod_dea(X)
    return pd.Series(ci, index=idx)


# ── Exposure: single BoD-DEA over all four indicators ────────────────────────────
print('Running BoD DEA for Exposure index ...')
exposure_composite = compute_composite({
    'pop_density':         pop_density,
    'cropland_ha_per_km2': cropland_density,
    'livestock_density':   livestock_series,
    'bws_score':           bws_series,
})
if evaluateDEA:
    exp_df = pd.DataFrame({
        'pop_density':         pop_density,
        'cropland_ha_per_km2': cropland_density,
        'livestock_density':   livestock_series,
        'bws_score':           bws_series,
    }).reindex(exposure_composite.index)
    dEmax = exp_df.max(axis=1)
    fig, ax = plt.subplots()
    ax.scatter(list(dEmax), exposure_composite)
    ax.set_xlabel('Maximum exposure indicator')
    ax.set_ylabel('DEA composite')
    ax.set_title("Evaluate exposure's DEA")
    plt.show()
print(f'  Exposure CI: min={exposure_composite.min():.4f}  max={exposure_composite.max():.4f}')
print('>>>>> Drought exposure is completed.')


# ── Vulnerability: two-step composite following Carrao et al. (2016) Eq. 3 ──────────
# Step 1: separate BoD-DEA per factor group (Social, Economic, Infrastructure)
# Step 2: dV = (Social + Economic + Infrast) / 3
# All three indicators are inverted (higher value -> lower vulnerability).
VULN_FACTORS = {
    'Social':   ({'hdi':                     hdi_series},    ['hdi']),
    'Economic': ({'gdp_pc':                  gdp_series},    ['gdp_pc']),
    'Infrast':  ({'road_density_km_per_km2': road_density},  ['road_density_km_per_km2']),
}

print('Running BoD DEA for Vulnerability index ...')
d_v = []
fac_composite = None
for fac_, (indicators, invert_list) in VULN_FACTORS.items():
    print(f">>>>> Analyzing the '{fac_}' factors")
    fac_composite = compute_composite(indicators, invert=invert_list)
    d_v.append(fac_composite.values)
    if evaluateDEA:
        dVmax = list(indicators.values())[0].reindex(fac_composite.index)
        fig, ax = plt.subplots()
        ax.scatter(list(minmax_norm(dVmax)), fac_composite)
        ax.set_xlabel(f'Maximum {fac_} vulnerability')
        ax.set_ylabel('DEA composite')
        ax.set_title(f"Evaluate vulnerability's DEA ({fac_})")
        plt.show()

vulnerability_composite = pd.Series(
    np.nanmean(np.array(d_v), axis=0),
    index=fac_composite.index
)
print(f'  Vulnerability CI: min={vulnerability_composite.min():.4f}  max={vulnerability_composite.max():.4f}')
print('>>>>> Drought vulnerability is completed.')

# Save composite indices
ci_df = pd.DataFrame({
    'exposure_raw':      exposure_composite,
    'vulnerability_raw': vulnerability_composite,
})
ci_df.index.name = 'district'
ci_df.to_csv(RISK_DIR / 'composite_indices_districts.csv')
print('Composite indices saved \u2192 composite_indices_districts.csv')

Running BoD DEA for Exposure index ...
  Exposure CI: min=0.0854  max=1.0000
>>>>> Drought exposure is completed.
Running BoD DEA for Vulnerability index ...
>>>>> Analyzing the 'Social' factors
>>>>> Analyzing the 'Economic' factors
>>>>> Analyzing the 'Infrast' factors
  Vulnerability CI: min=0.2597  max=0.9998
>>>>> Drought vulnerability is completed.
Composite indices saved → composite_indices_districts.csv


## Cell 6 — Compute risk per scenario

**Risk = Hazard × Exposure × Vulnerability**

Hazard values are used directly (already in [0, 1] from the hazard notebook output).  
Exposure and vulnerability are scenario-invariant (present-day conditions).

Risk classes are assigned with **Fisher-Jenks natural breaks** (5 classes), consistent with the hazard classification.

In [9]:
def compute_risk(
    hazard_scores: pd.Series,
    exposure: pd.Series,
    vulnerability: pd.Series
) -> pd.DataFrame:
    """
    Risk = hazard_raw x exposure_raw x vulnerability_raw.
    Hazard values are used directly (already in [0, 1] from the hazard notebook).
    Exposure and vulnerability are BoD DEA composite scores.
    """
    df = pd.DataFrame({
        'hazard_raw':        hazard_scores,
        'exposure_raw':      exposure.reindex(hazard_scores.index),
        'vulnerability_raw': vulnerability.reindex(hazard_scores.index),
    })
    df['risk_raw'] = (df['hazard_raw'] * df['exposure_raw'] * df['vulnerability_raw']).round(3)
    return df.reset_index().rename(columns={'index': 'district'})


def classify_risk(df: pd.DataFrame, n_classes: int = 5) -> pd.DataFrame:
    """
    Assign risk classes using Fisher-Jenks natural breaks.
    Falls back to equal-quantile breaks if jenkspy is not installed.
    """
    df    = df.copy()
    valid = df['risk_raw'].dropna().tolist()

    if HAS_JENKSPY and len(valid) >= n_classes:
        breaks = jenks_breaks(valid, n_classes=n_classes)
    else:
        breaks = list(np.quantile(valid, np.linspace(0, 1, n_classes + 1)))
        if not HAS_JENKSPY:
            print('  (Using quantile breaks \u2014 install jenkspy for Jenks natural breaks)')

    breaks[0]  -= 1e-9
    breaks[-1] += 1e-9
    df['risk_cat'] = pd.cut(
        df['risk_raw'], bins=breaks, labels=range(1, n_classes + 1), right=True
    ).astype('Int64')
    return df


# ── Align composite indices to hazard key space ────────────────────────────────────
exposure_final      = exposure_composite.copy()
vulnerability_final = vulnerability_composite.copy()

if exposure_final.index.duplicated().any():
    exposure_final = exposure_final[~exposure_final.index.duplicated(keep='first')]
if vulnerability_final.index.duplicated().any():
    vulnerability_final = vulnerability_final[~vulnerability_final.index.duplicated(keep='first')]

print(f'Exposure districts      : {len(exposure_final)}')
print(f'Vulnerability districts : {len(vulnerability_final)}')


# ── Risk calculation per scenario ─────────────────────────────────────────────────
risk_tables = {}

for scen in SCENARIOS:
    if scen not in hazard_all.columns:
        print(f'SKIP {scen} \u2014 not in hazard data')
        continue

    hz = hazard_all.set_index('district')[scen].dropna()
    if hz.index.duplicated().any():
        hz = hz[~hz.index.duplicated(keep='first')]

    rt = compute_risk(hz, exposure_final, vulnerability_final)
    rt = classify_risk(rt)
    risk_tables[scen] = rt

    top3 = rt.nlargest(3, 'risk_raw')
    print(f'\n{scen.upper()} \u2014 top 3 highest-risk districts:')
    print(top3[['district', 'hazard_raw', 'exposure_raw', 'vulnerability_raw', 'risk_raw']]
          .to_string(index=False))


# ── Save per-scenario CSVs ───────────────────────────────────────────────────────────
for scen, df in risk_tables.items():
    out = RISK_DIR / f'droughtrisk_KZ_{scen}_districts.csv'
    df.to_csv(out, index=False)
    print(f'Saved: {out.name}')
print('>>>>> Risk calculation completed.')

Exposure districts      : 227
Vulnerability districts : 227

HISTORIC — top 3 highest-risk districts:
                             district  hazard_raw  exposure_raw  vulnerability_raw  risk_raw
Mangystau District | Mangystau Region       0.848           1.0           0.872170     0.740
         Zhanaozen | Mangystau Region       0.886           1.0           0.829352     0.735
  Munaily District | Mangystau Region       0.842           1.0           0.866291     0.729

SSP126_NF — top 3 highest-risk districts:
                              district  hazard_raw  exposure_raw  vulnerability_raw  risk_raw
   Munaily District | Mangystau Region       0.833           1.0           0.866291     0.722
          Zhanaozen | Mangystau Region       0.870           1.0           0.829352     0.722
Tüpqarağan District | Mangystau Region       0.822           1.0           0.866949     0.713

SSP126_FF — top 3 highest-risk districts:
                              district  hazard_raw  exposure_raw

## Cell 7 — Summary table & district rankings

A long-format CSV combining all scenarios, suitable for further analysis or reporting.  
The console output shows the class distribution for the historical baseline and lists the highest-risk districts.

In [10]:
rows = []
for scen, tbl in risk_tables.items():
    for _, row in tbl.iterrows():
        rc    = int(row['risk_cat']) if pd.notna(row.get('risk_cat')) else None
        parts = str(row['district']).split(' | ')
        rows.append({
            'scenario':      scen,
            'district_name': parts[0],
            'oblast_name':   parts[1] if len(parts) > 1 else '',
            'district_key':  row['district'],
            'hazard':        round(float(row['hazard_raw']),        4),
            'exposure':      round(float(row['exposure_raw']),      4),
            'vulnerability': round(float(row['vulnerability_raw']), 4),
            'risk':          round(float(row['risk_raw']),          4),
            'risk_class':    rc,
            'risk_label':    RISK_LABELS[rc - 1] if rc else None,
        })

summary = pd.DataFrame(rows)
summary.to_csv(RISK_DIR / 'droughtrisk_KZ_summary_all_scenarios_districts.csv', index=False)
print('>>>>> Full summary saved \u2192 droughtrisk_KZ_summary_all_scenarios_districts.csv')

# ── Historical class distribution ────────────────────────────────────────────────────────
hist_sum = summary[summary['scenario'] == 'historic'].copy()
print('\nDistricts by risk class \u2014 HISTORICAL:')
print(
    hist_sum.groupby('risk_label', sort=False)['district_name']
    .count()
    .rename('n_districts')
    .reindex(RISK_LABELS)
    .to_string()
)

# ── Top 20 highest-risk districts (historical) ────────────────────────────────────────
print('\nTop 20 highest-risk districts (historical):')
top20 = (
    hist_sum
    .sort_values('risk', ascending=False)
    .head(20)
    [['district_name', 'oblast_name', 'hazard', 'exposure', 'vulnerability', 'risk', 'risk_label']]
)
print(top20.to_string(index=False))

>>>>> Full summary saved → droughtrisk_KZ_summary_all_scenarios_districts.csv

Districts by risk class — HISTORICAL:
risk_label
Very Low     55
Low          47
Medium       43
High         31
Very High    10

Top 20 highest-risk districts (historical):
        district_name      oblast_name  hazard  exposure  vulnerability  risk risk_label
   Mangystau District Mangystau Region   0.848    1.0000         0.8722 0.740  Very High
            Zhanaozen Mangystau Region   0.886    1.0000         0.8294 0.735  Very High
     Munaily District Mangystau Region   0.842    1.0000         0.8663 0.729  Very High
    Qaraqïya District Mangystau Region   0.835    1.0000         0.8736 0.729  Very High
  Tüpqarağan District Mangystau Region   0.830    1.0000         0.8669 0.720  Very High
    Ordabasy District Turkistan Region   0.852    0.8433         0.9777 0.702  Very High
    Zhetisay District Turkistan Region   0.720    0.9843         0.9510 0.674  Very High
Enbekshinsky district         Shymk

## Contributors

This Kazakhstan adaptation was developed building on the original CLIMAAX workflow by [Silvia Artuso](https://iiasa.ac.at/staff/silvia-artuso) and [Dor Fridman](https://iiasa.ac.at/staff/dor-fridman) from [IIASA’s Water Security Research Group](https://iiasa.ac.at/programs/biodiversity-and-natural-resources-bnr/water-security), supported by [Michaela Bachmann](https://iiasa.ac.at/staff/michaela-bachmann) from [IIASA’s Systemic Risk and Resilience Research Group](https://iiasa.ac.at/programs/advancing-systems-analysis-asa/systemic-risk-and-resilience).

## References

[1] Zargar, A., Sadiq, R., Naser, B., & Khan, F. I. (2011). A review of drought indices. *Environmental Reviews*, 19: 333–349.

[2] Carrão, H., Naumann, G., & Barbosa, P. (2016). Mapping global patterns of drought risk: An empirical framework based on sub-national estimates of hazard, exposure and vulnerability. *Global Environmental Change*, 39, 108–124.

[3] Cherchye, L., Moesen, W., Rogge, N., & Van Puyenbroeck, T. (2007). An introduction to ‘benefit of the doubt’ composite indicators. *Social Indicators Research*, 82(1), 111–145.

[4] Lyon, B., & Barnston, A. G. (2005). ENSO and the spatial extent of interannual precipitation extremes in tropical land areas. *Journal of Climate*, 18(23), 5095–5109.

[5] Carrão, H., Singleton, A., Naumann, G., Barbosa, P., & Vogt, J. V. (2014). An optimized system for the classification of meteorological drought intensity with applications in drought frequency analysis. *Journal of Applied Meteorology and Climatology*, 53(8), 1943–1960.